# Unit 4 — Notebook 2: Deploying and Scaling ML Containers with Kubernetes

**What you will learn:**
1. What problems Kubernetes (K8s) solves that Docker alone cannot
2. The key Kubernetes objects: Pod, Deployment, Service, HPA
3. How to write production-ready YAML manifests for an ML API
4. How to perform rolling updates and rollbacks with zero downtime

**Prerequisite:** Notebook 01 — you should have the `iris-api:v1` Docker image concept fresh in your mind.

**How to use:** `%%writefile` cells create real YAML files on disk. `subprocess` cells show `kubectl` commands — they run for real when a cluster is reachable; otherwise (kubectl missing, or installed with no cluster running) they print the expected output, clearly labeled.


---
## Section 1 — What Kubernetes Solves

Docker runs one container on one machine. That is fine for development. In production you need answers to questions Docker alone cannot answer:

- **Crash recovery:** Your container dies at 3 AM. Who restarts it?
- **Traffic spikes:** 10x the normal requests hit your API. How do you spin up more containers automatically?
- **Updates without downtime:** You have a new model. How do you swap it out while requests keep flowing?
- **Multiple machines:** Your service runs on a cluster of 20 nodes. How do you schedule containers across them?

Kubernetes answers all of these. It is a *container orchestration platform* — it manages where containers run, how many run, and what happens when they fail.

**Concrete example:** You deploy 3 replicas of `iris-api`. One pod crashes. Within seconds, Kubernetes detects the failure via the liveness probe, starts a replacement pod on a healthy node, and routes traffic only to the 2 healthy pods until the replacement is ready. No human intervention needed.


---
## Section 2 — Key Kubernetes Objects

| Object | Analogy | What it does |
|---|---|---|
| **Pod** | A single running process | Smallest deployable unit — usually 1 container plus shared network/storage |
| **Deployment** | A factory that makes pods | Declares how many replicas you want; replaces failed pods; manages rollouts |
| **Service** | A stable phone number | Gives pods a fixed DNS name; load-balances requests across all healthy pods |
| **HPA** | Auto-hiring / auto-firing | HorizontalPodAutoscaler: adds or removes pods based on CPU or custom metrics |

The usual hierarchy: a **Deployment** creates and owns **Pods**. A **Service** selects those Pods by label and routes traffic to them. An **HPA** watches metrics and tells the Deployment to scale up or down.


---
## Section 3 — Writing the YAML Manifests

Kubernetes is configured declaratively: you describe the desired state in YAML and `kubectl apply` makes it so. Each `%%writefile` below creates a real file in `k8s/`.


In [1]:
import os
os.makedirs('k8s', exist_ok=True)
print('Created k8s/ directory')


Created k8s/ directory


### 3a. Deployment

The Deployment tells Kubernetes: "keep 3 replicas of this pod running at all times". It also controls how rollouts happen — key fields are explained as inline comments.


In [2]:
%%writefile k8s/deployment.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: iris-api            # Name of this Deployment object
  labels:
    app: iris-api
spec:
  replicas: 3               # Run 3 identical pods at all times
  selector:
    matchLabels:
      app: iris-api         # This Deployment manages pods with this label
  strategy:
    type: RollingUpdate
    rollingUpdate:
      maxUnavailable: 1     # At most 1 pod can be down during an update
      maxSurge: 1           # At most 1 extra pod can exist during an update
  template:
    metadata:
      labels:
        app: iris-api       # Pods get this label — the selector above matches it
    spec:
      containers:
        - name: iris-api
          image: iris-api:v1            # Image built in Notebook 01
          imagePullPolicy: IfNotPresent # Use local image if already pulled
          ports:
            - containerPort: 8000
          resources:
            requests:
              cpu: "100m"         # 0.1 CPU cores guaranteed
              memory: "128Mi"     # 128 MB RAM guaranteed
            limits:
              cpu: "500m"         # Cannot exceed 0.5 CPU cores
              memory: "256Mi"     # Cannot exceed 256 MB RAM
          livenessProbe:
            # K8s calls /health every 10s.
            # If it fails 3 times in a row, the pod is killed and restarted.
            httpGet:
              path: /health
              port: 8000
            initialDelaySeconds: 5   # Wait 5s before first check (let app start)
            periodSeconds: 10
            failureThreshold: 3
          readinessProbe:
            # K8s calls /health every 5s.
            # Pod only receives traffic when this passes.
            # Unlike liveness: a not-ready pod is removed from the load balancer
            # but NOT restarted — it waits to recover on its own.
            httpGet:
              path: /health
              port: 8000
            initialDelaySeconds: 3
            periodSeconds: 5
            failureThreshold: 2


Overwriting k8s/deployment.yaml


### 3b. Service

A Service gives the pods a stable DNS name and load-balances requests across all healthy pods. Without a Service, pods have changing IP addresses every time they restart — clients would have no reliable address to connect to.


In [3]:
%%writefile k8s/service.yaml
apiVersion: v1
kind: Service
metadata:
  name: iris-api-svc
spec:
  type: LoadBalancer          # Cloud provider provisions an external IP
                              # Use NodePort for bare-metal / local clusters
  selector:
    app: iris-api             # Route traffic to pods with this label
  ports:
    - protocol: TCP
      port: 80                # External port clients connect to
      targetPort: 8000        # Port the container listens on


Overwriting k8s/service.yaml


### 3c. HorizontalPodAutoscaler

The HPA watches CPU usage across all pods. If the average exceeds 70%, it adds pods (up to 10). If CPU drops, it removes pods (down to 2). This handles traffic spikes without manual intervention.


In [4]:
%%writefile k8s/hpa.yaml
apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata:
  name: iris-api-hpa
spec:
  scaleTargetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: iris-api            # The Deployment to scale
  minReplicas: 2              # Never go below 2 pods (availability guarantee)
  maxReplicas: 10             # Never go above 10 pods (cost cap)
  metrics:
    - type: Resource
      resource:
        name: cpu
        target:
          type: Utilization
          averageUtilization: 70  # Scale out when avg CPU > 70%


Overwriting k8s/hpa.yaml


In [5]:
# Verify all three files were created
import os
for f in ['k8s/deployment.yaml', 'k8s/service.yaml', 'k8s/hpa.yaml']:
    size = os.path.getsize(f)
    print(f"{f}  ({size} bytes)")


k8s/deployment.yaml  (2003 bytes)
k8s/service.yaml  (454 bytes)
k8s/hpa.yaml  (533 bytes)


---
## Section 4 — kubectl Commands

These cells run `kubectl` via `subprocess`. If `kubectl` and a cluster are not available, the expected output is shown so you know what to look for in a real environment.


In [6]:
# Helper used by every kubectl cell below. It is honest about all three cases:
#   1. kubectl installed AND a cluster answers -> run for real, show real output
#   2. kubectl not installed                   -> FileNotFoundError
#   3. kubectl present but NO cluster running  -> command fails ("connection refused")
# Cases 2 and 3 print the expected output instead, clearly labeled — no raw
# error walls, no pretending a command succeeded.
import subprocess

def run_kubectl(args, expected_output=None, timeout=30):
    """Run a kubectl command; print a labeled teaching fallback if it cannot run."""
    cmd = ["kubectl"] + args
    print("Command:", " ".join(cmd))
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
        if result.returncode == 0:
            # Real cluster answered: show the genuine output
            print(result.stdout or "(no output)")
            print()
            return True
        print("(kubectl is installed but no cluster is reachable — expected output:)")
    except FileNotFoundError:
        print("(kubectl not installed — expected output:)")
    except subprocess.TimeoutExpired:
        print("(timed out — expected output:)")
    if expected_output:
        print(expected_output)
    print()
    return False

# Apply all three manifests at once — Deployment, Service, and HPA
run_kubectl(
    ["apply", "-f", "k8s/"],
    expected_output=(
        "deployment.apps/iris-api created\n"
        "service/iris-api-svc created\n"
        "horizontalpodautoscaler.autoscaling/iris-api-hpa created"
    )
)


Command: kubectl apply -f k8s/
(kubectl is installed but no cluster is reachable — expected output:)
deployment.apps/iris-api created
service/iris-api-svc created
horizontalpodautoscaler.autoscaling/iris-api-hpa created



False

In [7]:
# Inspect the running deployment (reuses the run_kubectl helper defined above)

# List pods — after ~30s all should be Running
run_kubectl(
    ["get", "pods", "-l", "app=iris-api"],
    expected_output=(
        "NAME                        READY   STATUS    RESTARTS   AGE\n"
        "iris-api-7d4f9c8b5-xkp2q   1/1     Running   0          45s\n"
        "iris-api-7d4f9c8b5-mn7rj   1/1     Running   0          45s\n"
        "iris-api-7d4f9c8b5-v9tw1   1/1     Running   0          45s"
    )
)

# Describe one pod — shows probes, resource limits, recent events
run_kubectl(
    ["describe", "pod", "-l", "app=iris-api"],
    expected_output=(
        "Name:         iris-api-7d4f9c8b5-xkp2q\n"
        "Liveness:   http-get http://:8000/health delay=5s timeout=1s period=10s\n"
        "Readiness:  http-get http://:8000/health delay=3s timeout=1s period=5s\n"
        "Events:\n"
        "  Normal  Scheduled  pod was successfully assigned to node\n"
        "  Normal  Pulled     Container image pulled\n"
        "  Normal  Started    Started container iris-api"
    )
)

# View recent logs from all pods with the app=iris-api label
run_kubectl(
    ["logs", "-l", "app=iris-api", "--tail=5"],
    expected_output=(
        "INFO:     Application startup complete.\n"
        "INFO:     Uvicorn running on http://0.0.0.0:8000"
    )
)

# Show the Service and its external IP
run_kubectl(
    ["get", "svc", "iris-api-svc"],
    expected_output=(
        "NAME           TYPE           CLUSTER-IP     EXTERNAL-IP     PORT(S)       AGE\n"
        "iris-api-svc   LoadBalancer   10.96.45.123   34.102.136.180  80:31234/TCP  2m"
    )
)


Command: kubectl get pods -l app=iris-api
(kubectl is installed but no cluster is reachable — expected output:)
NAME                        READY   STATUS    RESTARTS   AGE
iris-api-7d4f9c8b5-xkp2q   1/1     Running   0          45s
iris-api-7d4f9c8b5-mn7rj   1/1     Running   0          45s
iris-api-7d4f9c8b5-v9tw1   1/1     Running   0          45s

Command: kubectl describe pod -l app=iris-api
(kubectl is installed but no cluster is reachable — expected output:)
Name:         iris-api-7d4f9c8b5-xkp2q
Liveness:   http-get http://:8000/health delay=5s timeout=1s period=10s
Readiness:  http-get http://:8000/health delay=3s timeout=1s period=5s
Events:
  Normal  Scheduled  pod was successfully assigned to node
  Normal  Pulled     Container image pulled
  Normal  Started    Started container iris-api

Command: kubectl logs -l app=iris-api --tail=5
(kubectl is installed but no cluster is reachable — expected output:)
INFO:     Application startup complete.
INFO:     Uvicorn running on ht

False

---
## Section 5 — Rolling Update: Deploying a New Model Version

When you train a new model and build `iris-api:v2`, you update the image tag. Kubernetes replaces pods one at a time, keeping the service available throughout.

The `RollingUpdate` strategy in the Deployment controls this:
- `maxUnavailable: 1` — at most 1 pod is down at any moment (2 of 3 always serve traffic)
- `maxSurge: 1` — at most 1 extra pod exists temporarily (up to 4 pods during the rollout)

Readiness probes are critical here: a new pod only receives traffic after its readiness probe passes. If the new image is broken and the probe never passes, the rollout stalls — and the old pods stay up.


In [8]:
# Roll out a new model version by updating the image tag in place
# (no need to edit YAML for a single-field change)
run_kubectl(
    ["set", "image", "deployment/iris-api", "iris-api=iris-api:v2"],
    expected_output=(
        "deployment.apps/iris-api image updated\n"
        "\n"
        "Kubernetes then performs a rolling update:\n"
        "  1. Starts a new pod with iris-api:v2\n"
        "  2. Waits for its readiness probe to pass\n"
        "  3. Removes one old iris-api:v1 pod\n"
        "  4. Repeats until all 3 pods run v2\n"
        "  Traffic never drops to 0 pods."
    )
)


Command: kubectl set image deployment/iris-api iris-api=iris-api:v2


(kubectl is installed but no cluster is reachable — expected output:)
deployment.apps/iris-api image updated

Kubernetes then performs a rolling update:
  1. Starts a new pod with iris-api:v2
  2. Waits for its readiness probe to pass
  3. Removes one old iris-api:v1 pod
  4. Repeats until all 3 pods run v2
  Traffic never drops to 0 pods.



False

In [9]:
# Watch the rolling update progress until it completes
run_kubectl(
    ["rollout", "status", "deployment/iris-api"],
    expected_output=(
        "Waiting for deployment iris-api rollout to finish: 1 out of 3 new replicas updated...\n"
        "Waiting for deployment iris-api rollout to finish: 2 out of 3 new replicas updated...\n"
        "deployment iris-api successfully rolled out"
    ),
    timeout=60,
)


Command: kubectl rollout status deployment/iris-api


(kubectl is installed but no cluster is reachable — expected output:)
Waiting for deployment iris-api rollout to finish: 1 out of 3 new replicas updated...
Waiting for deployment iris-api rollout to finish: 2 out of 3 new replicas updated...
deployment iris-api successfully rolled out



False

---
## Section 6 — Rollback

If the new version has a bug (degraded accuracy, crashes on real inputs, failed smoke tests), you roll back to the previous version with one command. Kubernetes stores rollout history automatically.


In [10]:
# Rollback workflow (reuses the run_kubectl helper defined above)

# View rollout history — every revision the Deployment has been through
run_kubectl(
    ["rollout", "history", "deployment/iris-api"],
    expected_output=(
        "REVISION  CHANGE-CAUSE\n"
        "1         <none>   (iris-api:v1)\n"
        "2         <none>   (iris-api:v2)"
    )
)

# Rollback to the previous revision — one command, zero downtime
run_kubectl(
    ["rollout", "undo", "deployment/iris-api"],
    expected_output="deployment.apps/iris-api rolled back"
)

# Verify pods are running v1 again
run_kubectl(
    ["get", "pods", "-l", "app=iris-api"],
    expected_output=(
        "NAME                        READY   STATUS    RESTARTS   AGE\n"
        "iris-api-7d4f9c8b5-ab1cd   1/1     Running   0          30s\n"
        "iris-api-7d4f9c8b5-ef2gh   1/1     Running   0          25s\n"
        "iris-api-7d4f9c8b5-ij3kl   1/1     Running   0          20s"
    )
)


Command: kubectl rollout history deployment/iris-api
(kubectl is installed but no cluster is reachable — expected output:)
REVISION  CHANGE-CAUSE
1         <none>   (iris-api:v1)
2         <none>   (iris-api:v2)

Command: kubectl rollout undo deployment/iris-api


(kubectl is installed but no cluster is reachable — expected output:)
deployment.apps/iris-api rolled back

Command: kubectl get pods -l app=iris-api
(kubectl is installed but no cluster is reachable — expected output:)
NAME                        READY   STATUS    RESTARTS   AGE
iris-api-7d4f9c8b5-ab1cd   1/1     Running   0          30s
iris-api-7d4f9c8b5-ef2gh   1/1     Running   0          25s
iris-api-7d4f9c8b5-ij3kl   1/1     Running   0          20s



False

---
## Summary

Kubernetes gives you three things that Docker alone cannot:

1. **Self-healing** — Deployments with liveness probes automatically restart crashed pods. The desired replica count is always reconciled.
2. **Scaling** — The HPA adds and removes pods based on real traffic load, so you pay for what you use.
3. **Zero-downtime updates** — Rolling update strategy swaps pods one at a time. Readiness probes ensure traffic is never routed to an unready pod. Rollback is one command.

The workflow: write YAML manifests → `kubectl apply` → Kubernetes manages the rest.

**Next:** Notebook 03 compares AWS SageMaker, Azure ML, and GCP Vertex AI — so you know when to use self-managed K8s vs. a fully managed ML platform.


---
## Self-Check Questions

Answer without scrolling, then re-read the relevant section to check.

1. **What is the difference between a liveness probe and a readiness probe?**
   *(Hint: one triggers a restart; the other controls whether traffic is sent to the pod.)*

2. **What happens when you run `kubectl apply` on a Deployment that is already deployed with the same image?**
   *(Hint: Kubernetes is declarative — it computes a diff between desired and actual state.)*

3. **If a Deployment has 3 replicas and 1 pod crashes, what does Kubernetes do?**
   *(Hint: trace what happens after a liveness probe fails three times.)*
